# The SAETASS Unit System

Welcome to the first SAETASS tutorial! Before building grids or running simulations, we will learn how SAETASS handles **physical units**, since every other tutorial relies on it.

Astrophysical transport problems mix very different scales: parsecs and centimetres, megayears and seconds, GeV and erg. To make these mix-ups impossible, SAETASS requires every physical input to carry its units as an [Astropy Quantity](https://docs.astropy.org/en/stable/units/), and converts all of them to a single set of **canonical units** when they enter the package.

## 1. Canonical units

The canonical units are defined in the `saetass.units` module, which we import as `su`:

In [1]:
import astropy.constants as const
import astropy.units as u
import numpy as np

from saetass import units as su

print("Length:  ", su.LENGTH)
print("Time:    ", su.TIME)
print("Energy:  ", su.ENERGY)
print("Momentum:", su.MOMENTUM)

Length:   pc
Time:     Myr
Energy:   GeV
Momentum: GeV_c


Besides these base units, `saetass.units` defines the derived units of every quantity that appears in the transport equation, such as the velocity, the diffusion coefficient, the momentum loss rate or the source term:

In [2]:
for name in [
    "VELOCITY",
    "DIFFUSION_COEFFICIENT",
    "MOMENTUM_LOSS_RATE",
    "F_PS",
    "PSI_P",
    "PSI_E",
    "SOURCE_PSI_P",
]:
    print(f"{name:<22} {getattr(su, name)}")

VELOCITY               pc / Myr
DIFFUSION_COEFFICIENT  pc2 / Myr
MOMENTUM_LOSS_RATE     GeV_c / Myr
F_PS                   1 / (GeV_c3 pc3)
PSI_P                  1 / (GeV_c pc3)
PSI_E                  1 / (GeV pc3)
SOURCE_PSI_P           1 / (GeV_c Myr pc3)


In particular, the transported quantity is the **differential density** $\psi_p(r, p) = \frac{dn}{dp} = 4\pi p^2 f_\mathrm{ps}$, expressed in `PSI_P`, i.e. $\mathrm{pc^{-3}\,(GeV/c)^{-1}}$.

## 2. Creating Quantities

A Quantity is created by multiplying a number or a NumPy array by a unit. **Any unit with the right dimensions is accepted**: we can use the units constants of `saetass.units` or any other Astropy unit, and SAETASS takes care of the conversion.

In [3]:
radius = 10.0 * u.pc  # canonical units
distance = 2.0 * u.kpc  # any length unit works
D = 3e28 * u.cm**2 / u.s  # usual astrophysical units for diffusion
energies = np.logspace(0, 3, 4) * su.ENERGY  # arrays work too

print(distance.to(su.LENGTH))
print(D.to(su.DIFFUSION_COEFFICIENT))
print(energies)

2000.0 pc
99431.53210628688 pc2 / Myr
[   1.   10.  100. 1000.] GeV


### A note on momentum

Momenta are expressed in $\mathrm{GeV}/c$, which SAETASS provides as the named unit `su.MOMENTUM` (shown as `GeV_c`). Quantities built from the speed of light constant, such as `u.GeV / const.c`, are converted to it automatically:

In [4]:
p1 = 1.0 * su.MOMENTUM
p2 = (1.0 * u.GeV / const.c).to(su.MOMENTUM)
print(p1, p2)

# Total energy of a proton with momentum p = 1 GeV/c
E_tot = np.sqrt((p1 * const.c) ** 2 + (const.m_p * const.c**2) ** 2).to(u.GeV)
print(E_tot)

1.0 GeV_c 1.0 GeV_c
1.3712601918673453 GeV


## 3. Inputs are always validated

SAETASS never guesses units. Passing a bare number where a physical quantity is expected, or a Quantity with the wrong dimensions, raises an error instead of being silently interpreted in some implicit unit:

In [5]:
from saetass.grid import Grid

try:
    Grid.uniform(r_min=0.0, r_max=10.0, num_r_cells=100)
except TypeError as error:
    print("TypeError:", error)

try:
    Grid.uniform(r_min=0.0 * u.pc, r_max=10.0 * u.Myr, num_r_cells=100)
except u.UnitsError as error:
    print("UnitsError:", error)

TypeError: Argument 'r_min' to function 'uniform' has no 'unit' attribute. You should pass in an astropy Quantity instead.
UnitsError: Argument 'r_max' to function 'uniform' must be in units convertible to 'pc'.


## 4. Parameters of the physical operators

The same rule applies to the parameters of the physical operators that we will pass to the `Solver` in the next tutorials. Each operator declares the parameters it accepts together with their canonical units, which we can inspect through its `PARAM_SPECS`. Parameters with unit `None` are numerical options, such as the CFL number, and are given as plain numbers:

In [6]:
from saetass.solvers.diffusion_solver import DiffusionSolver

for name, spec in DiffusionSolver.PARAM_SPECS.items():
    print(f"{name:<20} unit: {str(spec.unit):<12} callable allowed: {spec.dynamic}")

D_values             unit: pc2 / Myr    callable allowed: True
psi_end              unit: 1 / (GeV_c pc3) callable allowed: True
boundary_condition   unit: None         callable allowed: False


Time-dependent parameters can be given as **callables**. They receive the time as a Quantity and must return a Quantity, so they are written in physical units as well:

In [7]:
def D_of_t(t):
    # The diffusion coefficient doubles every megayear
    return 1.0 * su.DIFFUSION_COEFFICIENT * 2 ** (t / u.Myr)


print(D_of_t(0.5 * u.Myr))

1.4142135623730951 pc2 / Myr


## 5. Results are Quantities

Results are returned as Quantities in canonical units as well, e.g. the distribution held by a `State` or its simulation time, so they can be converted to whatever units are convenient for analysis or plotting with `.to()` or `.to_value()`:

In [8]:
t = 0.25 * su.TIME
r = np.array([1.0, 2.0, 3.0]) * su.LENGTH
print(t.to(u.kyr))
print(r.to_value(u.lyr))  # bare numbers, e.g. for plotting

250.0 kyr
[3.26156378 6.52312755 9.78469133]


Internally, the numerical kernels only work with plain floating-point arrays in canonical units, so this unit handling has no impact on the performance of the time loop.

## Summary

In this tutorial, you learned that:
1. SAETASS uses a single set of **canonical units** (pc, Myr, GeV, GeV/c), defined in `saetass.units`.
2. Every physical input must be an **Astropy Quantity**, in any compatible unit.
3. Bare numbers or incompatible units are **rejected**, never silently assumed.
4. Operator parameters declare their units in `PARAM_SPECS`, and time-dependent ones are callables working with Quantities.
5. **Results** are Quantities too, ready to be converted to any unit.

In the next tutorial, we will use this unit system to build our first **Grid** and **State**!